# Orto — Notebook 03: ML Confidence Classifier & Routing

**Project:** Orto Neurosymbolic GEC Engine  
**Batch:** Batch F — NLP Capstone  

### Overview
This notebook demonstrates feature extraction from candidate edits, training a Scikit-Learn Logistic Regression confidence classifier, evaluating precision/recall curves, and calibrating decision boundaries.

In [ ]:
from orto.ml.features import FeatureExtractor, FEATURE_NAMES
from orto.ml.router import EditConfidenceClassifier
from orto.ml.train_router import train_and_serialize_router

print("Feature Names in Classification Vector:")
for idx, name in enumerate(FEATURE_NAMES):
    print(f"{idx+1}. {name}")

In [ ]:
# Train and serialize model bundle
bundle = train_and_serialize_router(
    output_path="data/models/edit_router.joblib",
    ngram_path="data/models/ngram_bigram.json"
)
print("Classifier training completed successfully.")
print(f"Training instances: {bundle['n_samples']}")
print(f"Feature count: {len(bundle['feature_names'])}")

In [ ]:
# Test Classifier on Real Sentence
from orto.llm.schemas import DiagnosticEdit, SpanCoordinate

classifier = EditConfidenceClassifier(model_path="data/models/edit_router.joblib")
text = "The box of old vinyl records were dropped."
edit = DiagnosticEdit(
    span=SpanCoordinate(start_char=29, end_char=33, original_text="were"),
    replacement="was",
    errant_type="R:VERB:SVA",
    linguistic_rule="Subject-Verb Agreement",
    explanation="Subject 'box' is singular.",
    counterfactual_example="The boxes were dropped.",
    confidence=0.95,
    critic_verified=True
)

results = classifier.classify_edits(text, [edit])
for r in results:
    print(f"Candidate: '{r.original_text}' -> '{r.replacement}'")
    print(f"Calibrated Validity Probability: {r.prob_valid:.4f}")
    print(f"Decision Accepted: {r.is_accepted}")